# 电商网站转化漏斗分析

## 业务背景
某电商网站积累了 10 万名访客的访问与转化数据，包含用户属性（新老客、性别、年龄）、流量属性
（来源渠道、设备、操作系统、市场），以及各页面到达情况（首页、列表页、商品页、支付页、确认页）。
网站整体转化率偏低，需定位用户在购买路径中的主要流失环节，识别不同流量与人群的转化差异，
为页面优化、流量投放和人群运营提供依据。

## 分析路径
1. 连接数据库
2. 数据清洗（建视图，处理缺失值）
3. 整体转化漏斗：各层人数、逐级转化率、累计转化率
4. 分维度转化：新老客、来源、设备、系统、市场、性别、年龄、浏览深度
5. 优化空间测算（量化杠杆）
6. 结论与建议、数据局限


In [48]:
# 漏斗前提校验：确认不存在跳步（每一步的人都必须出现在上一步）
sql_skip = """
SELECT
    SUM(confirmation_page = 1 AND payment_page = 0)  AS 跳到确认页,
    SUM(payment_page = 1      AND product_page = 0)  AS 跳到支付页,
    SUM(product_page = 1      AND listing_page = 0)  AS 跳到商品页,
    SUM(listing_page = 1      AND home_page = 0)     AS 跳到列表页
FROM v_behavior
"""
pd.read_sql(sql_skip, engine)


,跳到确认页,跳到支付页,跳到商品页,跳到列表页
0,0.0,0.0,0.0,0.0


In [49]:
import os
import pandas as pd
import numpy as np
from math import erfc, sqrt
from sqlalchemy import create_engine
from pyecharts.charts import Funnel, Bar, Line
from pyecharts import options as opts
from IPython.display import display

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    pass

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').split(':')[0]
DB_PORT = os.getenv('DB_PORT', '3306')
DB_NAME_FUNNEL = os.getenv('DB_NAME_FUNNEL', 'funnel_analysis')

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}'
    f'/{DB_NAME_FUNNEL}?charset=utf8mb4'
)
print('数据库连接成功')


数据库连接成功


In [50]:
# sex/device/os/source 中的 '0' 是未填写，统一替换为“未知”，再基于视图分析
sql_view = """
CREATE OR REPLACE VIEW v_behavior AS
SELECT
    new_user,
    CASE WHEN age > 90 THEN NULL ELSE age END AS age,
    CASE WHEN sex = '0' THEN '未知' ELSE sex END AS sex,
    market,
    CASE WHEN device = '0' THEN '未知' ELSE device END AS device,
    CASE WHEN operative_system = '0' THEN '未知'
         ELSE operative_system END AS operative_system,
    CASE WHEN source = '0' THEN '未知' ELSE source END AS source,
    total_pages_visited,
    home_page, listing_page, product_page, payment_page, confirmation_page
FROM user_behavior
"""
with engine.begin() as conn:
    conn.execute(sql_view)

print('清洗视图 v_behavior 创建成功')


清洗视图 v_behavior 创建成功


In [51]:
sql_funnel = """
SELECT
    SUM(home_page) AS 首页,
    SUM(listing_page) AS 列表页,
    SUM(product_page) AS 商品页,
    SUM(payment_page) AS 支付页,
    SUM(confirmation_page) AS 确认页
FROM v_behavior
"""
raw = pd.read_sql(sql_funnel, engine).iloc[0]

names = ['首页', '列表页', '商品页', '支付页', '确认页']
funnel_df = pd.DataFrame({'环节': names, '到达人数': raw.values.astype(int)})
funnel_df['逐级转化率%'] = [np.nan] + [
    round(funnel_df['到达人数'][i] / funnel_df['到达人数'][i-1] * 100, 2)
    for i in range(1, 5)]
funnel_df['累计转化率%'] = [
    round(x / funnel_df['到达人数'][0] * 100, 2)
    for x in funnel_df['到达人数']]

# 漏斗图
fn = (Funnel()
      .add('人数', [list(z) for z in zip(names, funnel_df['到达人数'])],
           sort_='none', gap=2,
           label_opts=opts.LabelOpts(formatter='{b}: {c}人'))
      .set_global_opts(title_opts=opts.TitleOpts(title='整体转化漏斗')))
display(fn.render_notebook())
funnel_df


,环节,到达人数,逐级转化率%,累计转化率%
0,首页,100000,NaN,100.00
1,列表页,73894,73.89,73.89
2,商品页,49618,67.15,49.62
3,支付页,6862,13.83,6.86
4,确认页,2404,35.03,2.40


**漏斗结论**：10 万访客最终仅 2,404 人成交，**整体转化率 2.4%**。各层逐级转化：首页→列表 73.89%、列表→商品 67.15%、**商品→支付仅 13.83%**、支付→确认 35.03%。前半段承接尚可，问题集中在 "商品页到支付"。

In [52]:
step_labels = [f'{names[i-1]}→{names[i]}' for i in range(1, 5)]
bar = (Bar()
       .add_xaxis(step_labels)
       .add_yaxis('逐级转化率%', funnel_df['逐级转化率%'].dropna().tolist(),
                  category_gap='45%')
       .set_global_opts(title_opts=opts.TitleOpts(title='各环节逐级转化率'),
                        yaxis_opts=opts.AxisOpts(name='转化率%')))
bar.render_notebook()


In [53]:
def dim_report(group_expr):
    sql = f"""
    SELECT
        {group_expr} AS 分组,
        COUNT(*) AS 人数,
        SUM(product_page) AS 商品页,
        SUM(payment_page) AS 支付页,
        SUM(confirmation_page) AS 确认页
    FROM v_behavior
    GROUP BY {group_expr}
    ORDER BY 人数 DESC
    """
    d = pd.read_sql(sql, engine)
    d['商品->支付%'] = (d['支付页'] / d['商品页'] * 100).round(1)
    d['整体转化率%'] = (d['确认页'] / d['人数'] * 100).round(2)
    d['支付->确认%'] = (d['确认页'] / d['支付页'] * 100).round(1)
    return d


In [54]:
def z_test_2prop(x1, n1, x2, n2):
    """两比例 Z 检验（双尾），返回 (p1, p2, z, p_value)"""
    if n1 == 0 or n2 == 0:
        return None
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return p1, p2, 0.0, 1.0
    z = (p1 - p2) / se
    return p1, p2, z, erfc(abs(z) / sqrt(2))     # 双尾 p = erfc(|z| / 根号2)


def sig_label(p):
    """p 值转显著性文字（用中文，避免星号被 Markdown 吃掉）"""
    if p < 0.001:
        return '极显著'
    if p < 0.01:
        return '显著'
    if p < 0.05:
        return '较显著'
    return '不显著'


def sig_report(g, label='', x_col='确认页', n_col='人数'):
    """分组表 vs 剔除自身后的大盘，做两比例 Z 检验"""
    g = g.reset_index(drop=True)
    X, N = g[x_col].sum(), g[n_col].sum()
    rows = []
    for _, r in g.iterrows():
        group_name = r.get('分组', r.iloc[0])
        x1, n1 = int(r[x_col]), int(r[n_col])
        x2, n2 = X - x1, N - n1                  # 对照组剔除自身，避免大组被稀释
        res = z_test_2prop(x1, n1, x2, n2)
        if res is None:
            continue
        p1, p2, z, p = res
        ok = (n1 * p1 >= 5) and (n1 * (1 - p1) >= 5)     # Z 检验适用性
        rows.append({
            '维度': label,
            '分组': group_name,
            '人数': n1,
            '转化率%': round(p1 * 100, 2),
            '大盘%': round(p2 * 100, 2),
            '相对差异%': round((p1 / p2 - 1) * 100, 1) if p2 else None,
            'z': round(z, 2),
            'p值': round(p, 4),
            '显著性': sig_label(p),
            '样本充足': '是' if ok else '否',
        })

    return pd.DataFrame(rows)

In [55]:
age_sql = """
CASE
    WHEN age IS NULL THEN '6.异常'
    WHEN age <= 24 THEN '1.<24'
    WHEN age <= 34 THEN '2.25-34'
    WHEN age <= 44 THEN '3.35-44'
    WHEN age <= 54 THEN '4.45-54'
    ELSE '5.55+'
END
"""

In [56]:
sig_all = pd.concat([
    sig_report(dim_report("device"),                                          '设备'),
    sig_report(dim_report("source"),                                          '来源'),
    sig_report(dim_report("sex"),                                             '性别'),
    sig_report(dim_report("CASE WHEN new_user = 1 THEN '新客' ELSE '老客' END"), '新老客'),
    sig_report(dim_report(age_sql),                                           '年龄'),
    sig_report(dim_report("operative_system"),                                '操作系统'),
    sig_report(dim_report("CAST(market AS CHAR)"), '市场'),
], ignore_index=True)


sig_all


,维度,分组,人数,转化率%,大盘%,相对差异%,z,p值,显著性,样本充足
0,设备,mobile,60364,2.27,2.60,-12.6,-3.30,0.0010,极显著,是
1,设备,desktop,39503,2.60,2.27,14.6,3.35,0.0008,极显著,是
2,设备,未知,133,1.50,2.41,-37.5,-0.68,0.4976,不显著,否
3,来源,Direct,56688,3.03,1.58,91.3,14.80,0.0000,极显著,是
4,来源,Seo,28337,1.55,2.74,-43.5,-11.10,0.0000,极显著,是
5,来源,Ads,14852,1.65,2.54,-34.9,-6.50,0.0000,极显著,是
6,来源,未知,123,1.63,2.40,-32.4,-0.56,0.5730,不显著,否
7,性别,Female,60554,3.02,1.46,107.7,15.81,0.0000,极显著,是
8,性别,Male,38673,1.45,3.01,-52.0,-15.71,0.0000,极显著,是
9,性别,未知,773,1.94,2.41,-19.4,-0.84,0.3983,不显著,是


In [57]:
new_old = dim_report("CASE WHEN new_user = 1 THEN '新客' ELSE '老客' END")
new_old

,分组,人数,商品页,支付页,确认页,商品->支付%,整体转化率%,支付->确认%
0,老客,68475,34314.0,5033.0,1941.0,14.7,2.83,38.6
1,新客,31525,15304.0,1829.0,463.0,12.0,1.47,25.3


**结论**：老客转化率 2.83%、新客 1.47%，老客1.93倍；商品→支付 14.7% vs 12.0%。老客价值显著更高，应加强老客召回与复购运营。支付→确认分化最剧：老客 38.6% vs 新客 25.3%（差 13.3pp），提示支付履约环节对新客存在结构性障碍

In [58]:
source_df = dim_report("source")
bar = (Bar()
       .add_xaxis(source_df['分组'].tolist())
       .add_yaxis('整体转化率%', source_df['整体转化率%'].tolist())
       .set_global_opts(title_opts=opts.TitleOpts(title='各流量来源转化率'),
                        yaxis_opts=opts.AxisOpts(name='转化率%')))
display(bar.render_notebook())
source_df


,分组,人数,商品页,支付页,确认页,商品->支付%,整体转化率%,支付->确认%
0,Direct,56688,28516.0,4246.0,1718.0,14.9,3.03,40.5
1,Seo,28337,13773.0,1706.0,439.0,12.4,1.55,25.7
2,Ads,14852,7267.0,904.0,245.0,12.4,1.65,27.1
3,未知,123,62.0,6.0,2.0,9.7,1.63,33.3


**结论**：直接访问转化率 3.03% 最高，Ads 1.65%、Seo 1.55%，约为直接流量的一半
（51%~55%）。但 Direct 占比 56.7%，是用户主动输入网址、收藏夹或搜品牌词进来的，
属品牌认知的结果而非可投放的手段。可操作路径有两条：一是优化 Ads/Seo 的落地页承接
（两者整体转化相近，但差距集中在商品→支付：12.4% vs Direct 14.9%）；
二是核算各渠道 CAC，把预算投向 ROI 可量化的渠道，而非向 Direct 倾斜。


In [59]:
device_df = dim_report("device")
os_df = dim_report("operative_system")
print("--设备--")
print(device_df.to_string())
print("--系统--")
print(os_df.to_string())


--设备--
        分组     人数      商品页     支付页     确认页  商品->支付%  整体转化率%  支付->确认%
0   mobile  60364  29899.0  4105.0  1373.0     13.7    2.27     33.4
1  desktop  39503  19651.0  2750.0  1029.0     14.0    2.60     37.4
2       未知    133     68.0     7.0     2.0     10.3    1.50     28.6
--系统--
        分组     人数      商品页     支付页    确认页  商品->支付%  整体转化率%  支付->确认%
0  windows  29700  14740.0  1985.0  686.0     13.5    2.31     34.6
1      iOS  28570  14176.0  1952.0  689.0     13.8    2.41     35.3
2  android  27328  13528.0  1902.0  623.0     14.1    2.28     32.8
3      mac   8552   4297.0   686.0  327.0     16.0    3.82     47.7
4    other   4446   2183.0   248.0   61.0     11.4    1.37     24.6
5    linux   1187    582.0    75.0   15.0     12.9    1.26     20.0
6       未知    217    112.0    14.0    3.0     12.5    1.38     21.4


 **结论**：设备维度差异统计显著但效应量极小——桌面 2.60% vs 移动 2.27%，仅相差 0.33pp（p=0.0010）。10 万样本下这种差距会显著，但业务价值有限，优先级低于商品→支付断崖。操作系统维度更明确：Windows / iOS / Android 三者之间无显著差异（p=0.206 / 0.921 / 0.116），说明主流系统不影响转化；仅 mac 显著偏高 3.82%（p<0.0001），可能对应高消费力人群，值得单独观察。Linux（1.26%）与 other（1.37%）虽经检验显著低于大盘，
但合计仅 5,633 人（5.6%），量级过小，不构成投放取舍依据；未知组（217 人）不显著，不单独下结论。


In [60]:
market_df = dim_report("CAST(market AS CHAR)")
print(market_df.to_string())


  分组     人数      商品页     支付页     确认页  商品->支付%  整体转化率%  支付->确认%
0  1  58094  29374.0  4055.0  1424.0     13.8    2.45     35.1
1  3  17890   7672.0  1002.0   308.0     13.1    1.72     30.7
2  2  17795   9104.0  1287.0   473.0     14.1    2.66     36.8
3  4   6221   3468.0   518.0   199.0     14.9    3.20     38.4


**结论**：市场 1 是流量主体，转化与大盘持平，市场 4（3.20%）、市场 2（2.66%）转化较高，市场 3 最低（1.72%）；

In [61]:
sex_df = dim_report("sex")
print(sex_df.to_string())

       分组     人数      商品页     支付页     确认页  商品->支付%  整体转化率%  支付->确认%
0  Female  60554  30444.0  4601.0  1830.0     15.1    3.02     39.8
1    Male  38673  18785.0  2212.0   559.0     11.8    1.45     25.3
2      未知    773    389.0    49.0    15.0     12.6    1.94     30.6


 **结论**：女性 3.02% 是男性 1.45% 的约 2 倍。

In [62]:
# 交叉验证：性别 × 新老客（检验两个效应是否独立，而非互为镜像）
cross = pd.read_sql("""
    SELECT CASE WHEN new_user = 1 THEN '新客' ELSE '老客' END AS 新老客,
           sex AS 性别,
           COUNT(*) AS 人数,
           SUM(payment_page) AS 到支付页,
           SUM(confirmation_page) AS 成交
    FROM v_behavior
    GROUP BY 1, 2 ORDER BY 1, 2
""", engine)
cross['到支付率%'] = (cross.到支付页 / cross.人数 * 100).round(2)
cross['整体转化率%'] = (cross.成交 / cross.人数 * 100).round(2)
cross

,新老客,性别,人数,到支付页,成交,到支付率%,整体转化率%
0,新客,Female,19022,1190.0,334.0,6.26,1.76
1,新客,Male,12257,624.0,125.0,5.09,1.02
2,新客,未知,246,15.0,4.0,6.10,1.63
3,老客,Female,41532,3411.0,1496.0,8.21,3.60
4,老客,Male,26416,1588.0,434.0,6.01,1.64
5,老客,未知,527,34.0,11.0,6.45,2.09


**结论**：性别与新老客两个效应各自独立成立，并非互为镜像——新客内部女性仍显著高于男性（1.76% vs 1.02%，约 1.7 倍），老客内部同样成立（3.60% vs 1.64%，约 2.2 倍）。但两者存在叠加放大效应：单看维度，"女性 3.02%""老客 2.83%"各自约为 2 倍差距；交叉后，新客男性（1.02%）与老客女性（3.60%）差距达 3.5 倍。人群运营应直接瞄准交叉后的高价值象限——老客女性（41,532 人、转化 3.60%，贡献 1,496 单、占大盘 62%），而非把"女性""老客"当成两个独立机会各讲一遍。

In [63]:
age_df = dim_report(age_sql)
bar = (Bar()
       .add_xaxis(age_df['分组'].tolist())
       .add_yaxis('整体转化率%', age_df['整体转化率%'].tolist())
       .set_global_opts(title_opts=opts.TitleOpts(title='各年龄段转化率'),
                        yaxis_opts=opts.AxisOpts(name='转化率%')))
display(bar.render_notebook())
age_df


,分组,人数,商品页,支付页,确认页,商品->支付%,整体转化率%,支付->确认%
0,2.25-34,43744,21828.0,3016.0,1096.0,13.8,2.51,36.3
1,1.<24,26527,13271.0,2171.0,1007.0,16.4,3.80,46.4
2,3.35-44,24556,12069.0,1455.0,295.0,12.1,1.20,20.3
3,4.45-54,4772,2262.0,209.0,6.0,9.2,0.13,2.9
4,5.55+,399,188.0,11.0,0.0,5.9,0.00,0.0
5,6.异常,2,0.0,0.0,0.0,NaN,0.00,NaN


**结论**：年龄不是平滑递减，而是「35 岁断崖」。<24 岁转化率 3.80% 为全场最高，
极显著高于大盘；25–34 岁 2.51% 与大盘无显著差异（p=0.065），不应归为劣势人群；
从 35 岁起断崖下滑——35–44 岁 1.20%、45–54 岁 0.13%，均极显著低于大盘。
说明问题不在「年轻人更爱买」，而在 35 岁以上用户的转化链路受阻，
需优先排查注册与支付流程对年长用户的门槛。

样本量提示：45 岁以上样本急剧缩小（45–54 共 4,772 人、仅 6 人成交；
55+ 共 399 人、0 人成交；另有 2 人年龄超过 90 岁已判为录入异常、
单列为「6.异常」组），统计功效不足，不足以支撑「放弃该人群」的投放决策，
需扩大数据后再判断。




In [64]:
depth_sql = """
CASE
    WHEN total_pages_visited <= 3 THEN '1.1-3'
    WHEN total_pages_visited <= 6 THEN '2.4-6'
    WHEN total_pages_visited <= 10 THEN '3.7-10'
    WHEN total_pages_visited <= 15 THEN '4.11-15'
    ELSE '5.16+'
END
"""
depth_df = dim_report(depth_sql)
bar = (Bar()
       .add_xaxis(depth_df['分组'].tolist())
       .add_yaxis('整体转化率%', depth_df['整体转化率%'].tolist())
       .set_global_opts(title_opts=opts.TitleOpts(title='浏览深度与转化率'),
                        yaxis_opts=opts.AxisOpts(name='转化率%')))
display(bar.render_notebook())
depth_df


,分组,人数,商品页,支付页,确认页,商品->支付%,整体转化率%,支付->确认%
0,1.1-3,35522,16494.0,72.0,0.0,0.4,0.00,0.0
1,2.4-6,33090,16866.0,2904.0,112.0,17.2,0.34,3.9
2,3.7-10,19258,9449.0,1177.0,261.0,12.5,1.36,22.2
3,4.11-15,7098,3651.0,993.0,545.0,27.2,7.68,54.9
4,5.16+,5032,3158.0,1716.0,1486.0,54.3,29.53,86.6


**结论**：转化率随浏览深度单调上升（1–3 页 0%、4–6 页 0.34%、7–10 页 1.36%、
11–15 页 7.68%、16+ 页 29.53%）。但需注意两点：
（1）1–3 页的 0% 属结构性不可转化——走完五级漏斗至少需 5 个页面，
     有效对比应从 4–6 页起；
（2）控制漏斗阶段后该梯度消失：只要未走到第 5 步，无论页数多少转化率
     均为 0；走到第 5 步则恒为 100%。说明页数并非独立的购买意向信号，
     而是漏斗进度（中介变量）的伴生结果。故此处仅作描述性呈现，
     不作为因果结论，实际触达应直接以漏斗阶段（如"到商品页未支付"）为触发条件。


In [65]:
# 锚点设定依据：以数据内部已达到的最优组为参考，而非主观赋值
n_product, n_pay, n_deal = funnel_df['到达人数'][2:5].tolist()
pay_to_deal = n_deal / n_pay

print(f'基线：商品->支付 {n_pay/n_product:.2%}，支付->确认 {pay_to_deal:.2%}\n')

# 内部基准：各维度实际最优组已达成的水平（均已在数据中实现）
def group_rate(mask, num, den):
    sub = pd.read_sql(f"SELECT SUM({den}) AS d, SUM({num}) AS n FROM v_behavior WHERE {mask}", engine)
    return sub['n'][0] / sub['d'][0]

anchor_p2p = {
    '老客组已达':  group_rate("new_user = 0", 'payment_page', 'product_page'),
    '女性组已达':  group_rate("sex = 'Female'", 'payment_page', 'product_page'),
    '最优组已达(<24岁)': group_rate("age <= 24", 'payment_page', 'product_page'),
}
anchor_pay2d = {
    '老客组已达':  group_rate("new_user = 0", 'confirmation_page', 'payment_page'),
    'Direct组已达': group_rate("source = 'Direct'", 'confirmation_page', 'payment_page'),
    '最优组已达(<24岁)': group_rate("age <= 24", 'confirmation_page', 'payment_page'),
}

print('=== 杠杆一：商品->支付 提升至内部基准 ===')
for name, tgt in anchor_p2p.items():
    line = []
    for tag, ratio in [('乐观', 1.0), ('中性', 0.8), ('悲观', 0.6)]:
        p2d = pay_to_deal * ratio
        add = (n_product * tgt - n_pay) * p2d
        line.append(f'{tag} 多{add:.0f}人 {add/n_deal*100:+.1f}%')
    print(f'{name:<18} {"  |  ".join(line)}')

print('\n=== 杠杆二：支付->确认 提升至内部基准 ===')
for name, tgt in anchor_pay2d.items():
    add = n_pay * tgt - n_deal
    print(f'{name:<18} 多{add:.0f}人  {add/n_deal*100:+.1f}%')

print('\n注1：锚点取自数据内部已实现的群组水平，属可比基准而非主观假设；')
print('     但各组存在结构差异（年龄、来源本身即与转化率相关），')
print('     全站提升至该水平需实际干预验证，此处仅测算可达上限。')
print('注2：两个杠杆存在交互效应，测算值不可直接相加。')


基线：商品->支付 13.83%，支付->确认 35.03%

=== 杠杆一：商品->支付 提升至内部基准 ===
老客组已达              乐观 多146人 +6.1%  |  中性 多117人 +4.8%  |  悲观 多87人 +3.6%
女性组已达              乐观 多223人 +9.3%  |  中性 多178人 +7.4%  |  悲观 多134人 +5.6%
最优组已达(<24岁)        乐观 多440人 +18.3%  |  中性 多352人 +14.6%  |  悲观 多264人 +11.0%

=== 杠杆二：支付->确认 提升至内部基准 ===
老客组已达              多242人  +10.1%
Direct组已达          多372人  +15.5%
最优组已达(<24岁)        多779人  +32.4%

注1：锚点取自数据内部已实现的群组水平，属可比基准而非主观假设；
     但各组存在结构差异（年龄、来源本身即与转化率相关），
     全站提升至该水平需实际干预验证，此处仅测算可达上限。
注2：两个杠杆存在交互效应，测算值不可直接相加。


# 结论与行动建议

### 核心判断
"前松后紧"型漏斗：10 万访客、近半数看到商品页，最终仅 2.4% 成交。
商品→支付（13.83%）是流失人数最多的环节（42,756 人），
但以数据内部已达水平为锚测算，其可达提升空间仅 2.5pp；
支付→确认（35.03%）虽流失人数较少（4,458 人），
但距离已达最优组有 3.5~11pp 空间。因此 优先级应按
「单位投入的可达增益」排序，而非按流失人数排序。
资源应优先投向支付履约环节。


### 行动建议（按优先级）
1. 优先修复支付履约（支付→确认 35.03%）：该环节新老客分化最剧（老客 38.57% vs 新客 25.31%，差 13.25pp），提示存在支付方式缺失、
   强制注册、风控拦截等障碍。提升至老客已达水平即 +10.1%，
   提升至 Direct 组水平为 +15.5%。优先级最高。
2. 商品页→支付（13.83%）为最大流失池但提升空间有限：提升至最优组
   已达的 16.36% 方可获得 +11.0%~+18.3%。建议先小范围 A/B 验证
   再决定是否大额投入，不宜作为首要目标。
3. 流量结构：Direct 转化 3.03% 最高，说明品牌认知带来的流量质量最好——但它是结果不是手段，无法直接投放。可操作的是优化 Ads（1.65%）与 Seo（1.55%）的落地页承接：两者的差距集中在商品→支付（12.4% vs Direct 14.9%），应在落地页到商品页的衔接上做 A/B 验证；同时核算各渠道 CAC，把预算投向 ROI 可量化的渠道。
4. 聚焦高价值人群：高转化特征集中叠加——老客 + 女性 + <24 岁高度重叠（女性占新老客两侧人群均约 60%，老客女性单象限贡献 62% 成交），运营应直接瞄准交叉象限（老客女性、年轻老客），而非把性别、新老客、年龄当作三个独立机会分别投放。45 岁以上样本过小，暂不纳入投放取舍，需扩大数据后再判断。
5. 意向触发：以漏斗阶段为触发条件——对「已到商品页但未进入支付页」的人群，做精准优惠或召回（该环节流失 86.17%，是最大可干预池），并设对照组验证。

### 数据局限
- 用户级聚合、无访问时间戳，无法做时序/留存与访问时长分析
- 无订单金额、成本字段，无法直接计算 GMV 与广告 ROI，仅以转化率评估
- 公开教学数据集，结论用于方法论验证，具体收益需在真实业务数据上复测
